# Backbone Experiments
Run multiple backbone experiments with identical training/inference pipeline.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
from src.utils import get_device, set_seed
from src.models import ArcFaceModel
from src.training import fit
from src.wandb_utils import init_wandb

# Load environment variables from .env file
load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device

In [ ]:
!wandb login

## Base Config


In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("checkpoints"),

    # Model
    "backbone_name": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "freeze_backbone": True,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 50,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,

    # Reproducibility
    "seed": RANDOM_SEED,

    # Normalization
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

## Load Data


In [ ]:
train_df = data.load_train_df(config["data_dir"])
train_df, label_encoder = data.encode_labels(train_df)
num_classes = len(label_encoder.classes_)

train_data, val_data = data.train_val_split(
    train_df,
    val_split=config["val_split"],
    seed=config["seed"],
    stratify_col="ground_truth",
)

train_loader, val_loader = data.create_dataloaders(
    train_data,
    val_data,
    img_dir=config["data_dir"] / "train" / "train",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

pairs_df = data.load_test_pairs_df(config["data_dir"])
unique_images = set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique())
unique_images = sorted(list(unique_images))

test_df = pd.DataFrame({"filename": unique_images})

test_loader = data.create_test_loader(
    test_df,
    img_dir=config["data_dir"] / "test" / "test",
    input_size=config["input_size"],
    batch_size=config["batch_size"],
    num_workers=config["num_workers"],
    mean=config["mean"],
    std=config["std"],
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## Submission Helper


In [ ]:
def create_submission(
    model,
    device,
    pairs_df,
    test_loader,
    checkpoint_path,
    output_path,
):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    names, embeddings = inference.extract_embeddings_with_names(model, test_loader, device)
    embedding_lookup = inference.build_embedding_lookup(names, embeddings)

    similarities = inference.compute_similarity_for_pairs(pairs_df, embedding_lookup)

    submission_df = pd.DataFrame({
        "row_id": pairs_df["row_id"],
        "similarity": similarities,
    })

    submission_df.to_csv(output_path, index=False)
    return submission_df


## Run Experiments


In [ ]:
backbones = [
    "hf-hub:BVRA/MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-L-384",
    "hf-hub:BVRA/MegaDescriptor-L-384",
]


def run_experiment(backbone_name, run_idx):
    print("=" * 80)
    print(f"Run {run_idx}/{len(backbones)}: {backbone_name}")

    # Update config for this run
    config["backbone_name"] = backbone_name

    # Init model
    model = ArcFaceModel(
        backbone_name=config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        margin=config["arcface_margin"],
        scale=config["arcface_scale"],
        dropout=config["dropout"],
        pretrained=True,
        freeze_backbone=config["freeze_backbone"],
    ).to(device)

    param_count = sum(p.numel() for p in model.parameters())

    # W&B init
    run_name = f"{backbone_name.split('/')[-1]}-arcface-{run_idx}"
    wandb = init_wandb(config, run_name=run_name, param_count=param_count)

    # Training components
    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    # Train
    checkpoint_name = f"arcface_best_{run_idx}.pth"
    results = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    # Submission
    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    submission_path = config["checkpoint_dir"] / f"submission_{run_idx}.csv"
    submission_df = create_submission(
        model,
        device,
        pairs_df,
        test_loader,
        checkpoint_path,
        submission_path,
    )

    # W&B artifacts
    model_artifact = wandb.Artifact(
        name=f"arcface-model-{run_idx}",
        type="model",
        description="ArcFace model checkpoint",
    )
    model_artifact.add_file(str(checkpoint_path))
    wandb.log_artifact(model_artifact)

    submission_artifact = wandb.Artifact(
        name=f"submission-{run_idx}",
        type="submission",
        description="Competition submission file",
    )
    submission_artifact.add_file(str(submission_path))
    wandb.log_artifact(submission_artifact)

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, backbone_name in enumerate(backbones, start=1):
    run_experiment(backbone_name, run_idx)
